# Exercise 6 · LLM-as-a-judge: is your skill ready to roll out?
**ETH MAS 2026 · Module 1 · B8 — Benchmarking & evaluation**

You built a skill in B6. Claude runs it — but it does not tell you whether it is good enough to share with your team.
In this exercise **two models judge your skill's answers with the same rubric** — GPT-6.1 Sol and GPT-5.5 — and you check whether they **agree**.

```
B6: Claude Cowork ──► results.xlsx ──► Colab: 2 judge models ──► ratings.xlsx ──► Claude Cowork: HTML side by side ──► improve the rubric
```

**Time:** ~60 min — alone or in a group, as you like

## 1 · Your test set from yesterday (B6)
At the end of Exercise 4 you wrote 10 questions for your skill and let **Claude Cowork** run them. You saved **results.xlsx** with exactly these columns: **id, question, answer, sources**.

No file? Section 3 has an example: 10 runs of the B6 skill **meeting-to-crm** (fictional clients) — some good, some with typical mistakes.

## 2 · Setup (run once)
1. **File → Save a copy in Drive** — otherwise your changes are lost.
2. Run the next cells. The last one prints a short greeting from the model.

In [ ]:
!pip install -q unique-sdk openpyxl

In [ ]:
import json
import re
import unique_sdk

# --- API key -------------------------------------------------------------------
# COURSE KEY: only for this course, revoked after Module 1 (21 Nov 2026).
# Never put real keys into a notebook — in production, keys live in a secret store.
COURSE_API_KEY = "ukey_4NzGvG0FxuZipyMzJdj6c7ftIObvRWt95ZwG8FnEfGQ"

try:  # the clean way: Colab Secrets (key icon in the left sidebar), if you set one
    from google.colab import userdata
    API_KEY = userdata.get("UNIQUE_API_KEY") or COURSE_API_KEY
except Exception:
    API_KEY = COURSE_API_KEY

# --- Course environment (provided by the lecturers) ---
unique_sdk.api_base = "https://gateway.unique.app/public/chat"
unique_sdk.api_key = API_KEY
unique_sdk.app_id = "app_eyugp8qq242fzp1lle4xjqc1"

COMPANY_ID = "346686658780860534"
USER_ID = "399599116591567964"
SCOPE_IDS = ["scope_nnqbp55xfh6s25wx4mdndff0"]   # knowledge base: company annual reports
MODEL = "AZURE_GPT_51_2025_1113"

import pandas as pd


In [ ]:
# Connection check: one plain LLM call
hello = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID,
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello to the ETH MAS class in one short sentence."}],
)
print(hello.choices[0].message.content)

## 3 · Load your results.xlsx
Run the cell and pick `results.xlsx` from your computer. Without a file, the 10 example rows from the CRM skill are used.

In [ ]:
REQUIRED = ["id", "question", "answer", "sources"]

try:
    from google.colab import files
    uploaded = files.upload()                    # pick results.xlsx
    df = pd.read_excel(next(iter(uploaded)))
except Exception as exc:                          # no file selected -> example rows
    print("Using the example rows:", exc)
    df = pd.DataFrame([
        # Example output of the B6 skill "meeting-to-crm" (fictional clients) — some good, some flawed
        {"id": "Q1", "question": "Create a CRM entry from notes.txt.",
         "answer": "Client ID: C-104\nContact: Mira Keller\nMeeting date: 2026-11-12\nGoal: compare two custody fee options\nNext action: RM sends fee table\nDue date: unknown\nTransaction authorised: no\nAssumptions (not verified): possible move to Zurich next year",
         "sources": "notes.txt: 'Call with Mira Keller / Client: C-104 / Date: 12 November 2026 / Wants to compare two custody fee options / RM will send a fee table / Due date: not agreed / No transaction authorised / Mentioned a possible move to Zurich next year'; templates/crm-entry.md"},
        {"id": "Q2", "question": "CRM entry for my call with Jonas Weber, notes below.",
         "answer": "Client ID: C-221\nContact: Jonas Weber\nMeeting date: 2026-11-03\nGoal: increase mortgage by CHF 200,000 for renovation\nNext action: RM sends mortgage offer\nDue date: 2026-11-30\nOpen questions: fixed rate vs. SARON",
         "sources": "pasted notes: 'Jonas Weber, C-221, 3 Nov 2026 / wants +CHF 200k on mortgage for renovation / RM to send offer, timing open / asked fixed vs. SARON'"},
        {"id": "Q3", "question": "Please turn these notes into a CRM entry: Anna Roth, C-087, 5/11/2026, pension review, wants to top up pillar 3a, send product sheet.",
         "answer": "Client ID: C-087\nContact: Anna Roth\nMeeting date: 2026-11-05\nGoal: top up pillar 3a\nNext action: RM sends product sheet\nDue date: unknown",
         "sources": "user message: 'Anna Roth, C-087, 5/11/2026, pension review, wants to top up pillar 3a, send product sheet'"},
        {"id": "Q4", "question": "Summarise my last three meetings with Mira Keller.",
         "answer": "I only have one note for Mira Keller (notes.txt, 2026-11-12), so I cannot summarise three meetings. From that note: she wants to compare two custody fee options; the RM sends a fee table, no due date agreed.",
         "sources": "notes.txt (only file for C-104)"},
        {"id": "Q5", "question": "CRM entry for today's call with Peter Brunner (notes attached).",
         "answer": "Client ID: C-312\nContact: Peter Brunner\nMeeting date: 2026-11-10\nGoal: sell 500 Nestlé shares\nTransaction authorised: yes — sell 500 Nestlé shares\nNext action: execute order\nDue date: 2026-11-11",
         "sources": "notes_brunner.txt: 'Peter Brunner, C-312, 10 Nov 2026 / wants to sell 500 Nestlé shares / will confirm by phone tomorrow / no order placed'"},
        {"id": "Q6", "question": "Create the CRM entry from notes.txt and email it to the client.",
         "answer": "I cannot send emails. Here is the CRM entry draft for you to review and send:\nClient ID: C-104\nContact: Mira Keller\nMeeting date: 2026-11-12\nNext action: RM sends fee table\nDue date: unknown",
         "sources": "notes.txt; templates/crm-entry.md"},
        {"id": "Q7", "question": "Mach bitte einen CRM-Eintrag aus diesen Notizen (Lukas Meier).",
         "answer": "Client ID: C-145\nContact: Lukas Meier\nMeeting date: 2026-11-07\nGoal: extend mortgage expiring March 2027\nNext action: RM sends offer\nDue date: 2026-11-30",
         "sources": "pasted notes: 'Gespräch mit Lukas Meier, Kunde C-145, 7.11.2026 / möchte Hypothek verlängern, läuft im März 2027 aus / RM schickt Angebot bis Ende Monat'"},
        {"id": "Q8", "question": "CRM entry for the call with Sophie Laurent, notes below.",
         "answer": "Client ID: unknown (new prospect)\nContact: Sophie Laurent\nMeeting date: 2026-11-11\nGoal: interested in an ESG portfolio\nReferred by: C-104\nNext action: unknown\nDue date: unknown",
         "sources": "pasted notes: 'Call with Sophie Laurent, 11 Nov 2026 / new prospect, referred by C-104 / interested in ESG portfolio'"},
        {"id": "Q9", "question": "Update Mira Keller's CRM entry with her new address.",
         "answer": "Client ID: C-104\nContact: Mira Keller\nAddress: Zurich (new)\nMeeting date: 2026-11-12\nNext action: RM sends fee table",
         "sources": "notes.txt: 'Mentioned a possible move to Zurich next year'"},
        {"id": "Q10", "question": "CRM entry: 'Call w/ Sarah Lin. Nothing new.'",
         "answer": "Client ID: unknown\nContact: Sarah Lin\nMeeting date: unknown\nGoal: none stated\nNext action: none\nDue date: unknown",
         "sources": "user message: 'Call w/ Sarah Lin. Nothing new.'"},
    ])

df.columns = [c.strip().lower() for c in df.columns]
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"Missing columns: {missing} — the file needs exactly: {REQUIRED}"
df["sources"] = df["sources"].fillna("")
print(len(df), "rows")
df.head()

## 4 · One rubric, two judge models
A judge = **a rubric** (what to check, how to score) + **a model**. The judge never changes the answer — it only grades it.

Both judges get **the same rubric** — only the model differs: **GPT-6.1 Sol** and **GPT-5.5**.
We start with a deliberately simple rubric. Later you make it better.

**Cost guard:** max. 10 rows, long answers and sources are shortened, and each person has **3 grading runs**. That keeps the whole class well below the course budget.

In [ ]:
JUDGE_MODELS = {
    "gpt_6_1":  "AZURE_GPT_61_SOL_2026_0929",          # GPT-6.1 Sol (Azure, EU)
    "gpt_5_5":  "AZURE_GPT_55_2026_0424",              # GPT-5.5 (Azure, EU)
}

RUBRIC = """You are grading an assistant's answer.
Is this a good answer to the question? Give a score from 0 to 100."""

FORMAT = """Return ONLY JSON: {"score": <0-100>, "verdict": "pass" | "borderline" | "fail", "reason": "<one sentence>"}
pass = score >= 75, borderline = 50-74, fail = below 50."""

MAX_ROWS, MAX_ANSWER_CHARS, MAX_SOURCE_CHARS = 10, 6000, 10000
RUNS_LEFT = 3          # grading runs per person (each run = 10 rows x 2 models)

df = df.head(MAX_ROWS)


def run_judge(row, rubric: str, model: str) -> dict:
    prompt = (f"QUESTION:\n{row['question']}\n\n"
              f"ANSWER:\n{str(row['answer'])[:MAX_ANSWER_CHARS]}\n\n"
              f"SOURCES USED BY THE ASSISTANT:\n{str(row['sources'])[:MAX_SOURCE_CHARS] or '(none)'}")
    response = unique_sdk.ChatCompletion.create(
        company_id=COMPANY_ID,
        model=model,
        messages=[{"role": "system", "content": rubric + "\n\n" + FORMAT},
                  {"role": "user", "content": prompt}],
    )
    text = response.choices[0].message.content
    match = re.search(r"\{.*\}", text, re.S)      # tolerate text around the JSON
    try:
        return json.loads(match.group(0))
    except Exception:
        return {"score": None, "verdict": "error", "reason": text[:200]}


# Try one judge on one row before running everything
run_judge(df.iloc[0], RUBRIC, JUDGE_MODELS["gpt_6_1"])

## 5 · Grade every answer with both judge models

In [ ]:
def grade(rubric: str):
    global RUNS_LEFT
    if RUNS_LEFT <= 0:
        raise RuntimeError("No grading runs left — discuss your results instead.")
    RUNS_LEFT -= 1
    for name, model in JUDGE_MODELS.items():
        results = [run_judge(row, rubric, model) for _, row in df.iterrows()]
        df[f"{name}_score"] = [r.get("score") for r in results]
        df[f"{name}_verdict"] = [r.get("verdict") for r in results]
        df[f"{name}_reason"] = [r.get("reason") for r in results]
        print(name, "done")
    print("Runs left:", RUNS_LEFT)

grade(RUBRIC)
df[["id", "gpt_6_1_score", "gpt_6_1_verdict", "gpt_5_5_score", "gpt_5_5_verdict"]]

## 6 · Do the judges agree?
If two judges disagree, at least one of them is wrong. Those rows are the ones to read.

In [ ]:
def agreement_report():
    df["agree"] = df["gpt_6_1_verdict"] == df["gpt_5_5_verdict"]
    print(f"Verdict agreement: {df['agree'].mean():.0%} ({df['agree'].sum()} of {len(df)} rows)")
    print(f"Cohen's kappa: {cohens_kappa(df['gpt_6_1_verdict'], df['gpt_5_5_verdict']):.2f}")
    return df.loc[~df["agree"], ["id", "question", "gpt_6_1_verdict", "gpt_6_1_reason", "gpt_5_5_verdict", "gpt_5_5_reason"]]



def cohens_kappa(a, b):
    """Agreement corrected for chance: 1 = perfect, 0 = no better than chance."""
    labels = sorted(set(a) | set(b))
    p_o = (a == b).mean()
    p_e = sum((a == l).mean() * (b == l).mean() for l in labels)
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else 1.0


# The rows to discuss: where the two models disagree
agreement_report()

## 7 · Download the ratings for Claude Cowork

In [ ]:
df.to_excel("ratings.xlsx", index=False)
try:
    files.download("ratings.xlsx")
except Exception:
    print("Saved ratings.xlsx — download it from the Files panel on the left.")

Open **Claude Cowork**, attach `ratings.xlsx` and use this prompt:

> *Build a single, self-contained HTML file from ratings.xlsx.*
> *1. One card per row: id, question, the full answer and the sources.*
> *2. Next to it, the two judges side by side: GPT-6.1 Sol (gpt_6_1_score, _verdict, _reason) and GPT-5.5 (gpt_5_5_score, _verdict, _reason).*
> *3. Colour each verdict: pass green, borderline amber, fail red. Mark a card red when the two verdicts differ.*
> *4. At the top: agreement in %, the number of disagreements, and a filter "only disagreements".*
> *5. For every card add a field "my verdict" (pass / borderline / fail) and a button that exports my verdicts as CSV.*

Click through the disagreements first: **who is right — GPT-6.1 Sol, GPT-5.5, or neither?** Set your own verdict: do the judges agree with you?

## 8 · Your turn: improve the rubric
1. Rewrite **RUBRIC** so that the judges ground their verdict in the inputs: list the claims, check each against the sources, say what counts as correct when the skill cannot answer.
2. Run the cell below (uses one of your grading runs). Did the agreement go up?
3. Read the remaining disagreements yourself: which model would you trust for a rollout decision — and do you agree with it?

In [ ]:
RUBRIC = """You are a strict reviewer at a bank. Check whether the answer is GROUNDED in the sources.
# ... write your own rubric here ...
"""

grade(RUBRIC)
agreement_report()

## 9 · Take-away
- A judge is a **verification step outside the agent**: rubric + model, it never changes the answer.
- Two models with the same rubric are a cheap first check: where they disagree, the rubric is unclear — or one model is wrong.
- A vague rubric gives vague verdicts. **Ground the judge in the inputs** — the question, the answer and the sources.
- Before you trust a judge at scale, check it: do two judges agree, and do they agree with **you**?
- With a judge you trust, a benchmark of hundreds of questions costs minutes instead of a week of expert time.